In [ ]:
# =========================
# 1. CLONE REPO
# =========================
!git clone https://github.com/fbocchi/flickr30k.git
%cd flickr30k


# =========================
# 2. MOUNT GOOGLE DRIVE
# =========================
from google.colab import drive
drive.mount('/content/drive')


# =========================
# 3. PATH DATASET ZIP
# =========================
ZIP_PATH = "/content/drive/MyDrive/Colab Notebooks/flickr30k-dataset.zip"


# =========================
# 4. UNZIP DATASET
# =========================
!unzip -q "$ZIP_PATH" -d /content/


# =========================
# 5. MOVE INTO REPO
# =========================
!mv /content/flickr30k-dataset ./flickr30k-dataset


# =========================
# 6. CHECK STRUCTURE
# =========================
!ls flickr30k-dataset | head

print("\nSetup completato ✔")

In [ ]:
import os
import numpy as np
import tensorflow as tf
import h5py
import keras
from keras.applications import VGG16
from keras.applications.vgg16 import preprocess_input
from keras.preprocessing import image
from tqdm import tqdm

# =========================
# CONFIG
# =========================
IMAGE_DIR = "/content/flickr30k/flickr30k-images"
OUTPUT_FILE = "/content/drive/MyDrive/Colab Notebooks/flickr30k_vgg16_features.h5"
BATCH_SIZE = 32
IMG_SIZE = (224, 224)

# =========================
# MODEL
# =========================
base_model = VGG16(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

model = keras.Sequential([
    base_model,
    keras.layers.GlobalAveragePooling2D()
])

# =========================
# IMAGE PATHS
# =========================
image_paths = sorted([
    os.path.join(IMAGE_DIR, f)
    for f in os.listdir(IMAGE_DIR)
    if f.lower().endswith(".jpg")
])

print(f"Found {len(image_paths)} images")

# =========================
# CREATE HDF5 FILE
# =========================
with h5py.File(OUTPUT_FILE, "w") as h5f:

    for i in tqdm(range(0, len(image_paths), BATCH_SIZE)):
        batch_paths = image_paths[i:i+BATCH_SIZE]

        batch_imgs = []
        batch_names = []

        for p in batch_paths:
            try:
                img = image.load_img(p, target_size=IMG_SIZE)
                x = image.img_to_array(img)
                batch_imgs.append(x)
                batch_names.append(os.path.basename(p))
            except Exception as e:
                print(f"Error: {p} -> {e}")

        if not batch_imgs:
            continue

        batch_imgs = np.array(batch_imgs, dtype=np.float32)
        batch_imgs = preprocess_input(batch_imgs)

        batch_feats = model.predict(batch_imgs, verbose=0)

        # save directly to disk (NO RAM DICT)
        for name, feat in zip(batch_names, batch_feats):
            h5f.create_dataset(
                name,
                data=feat.astype(np.float32),
                compression="gzip"
            )

print("Saved HDF5 features ✔")

In [ ]:
import h5py

data = h5py.File("/content/drive/MyDrive/Colab Notebooks/flickr30k_vgg16_features.h5", "r")

feat = data["1000092795.jpg"][:] # type: ignore
print(feat.shape)  # type: ignore # (512,)